In [14]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

In [15]:
from pathlib import Path

# Project and data paths
PROJECT_ROOT = Path.cwd().parent
TRAIN_DIR = PROJECT_ROOT / "data" / "raw" / "train"
TEST_DIR = PROJECT_ROOT / "data" / "raw" / "test"

print("Project:", PROJECT_ROOT)
print("Train folder:", TRAIN_DIR)
print("Test folder:", TEST_DIR)

Project: c:\Users\nihar\OneDrive\Attachments\AmazonML-26
Train folder: c:\Users\nihar\OneDrive\Attachments\AmazonML-26\data\raw\train
Test folder: c:\Users\nihar\OneDrive\Attachments\AmazonML-26\data\raw\test


In [16]:
# Read a small sample from each training source
source1_sample = pd.read_csv(
    TRAIN_DIR / "train_source1.tsv",
    sep="\t",
    nrows=5,
    encoding="utf-8"
)

source2_sample = pd.read_csv(
    TRAIN_DIR / "train_source2.tsv",
    sep="\t",
    nrows=5,
    encoding="utf-8"
)

source3_sample = pd.read_csv(
    TRAIN_DIR / "train_source3.tsv",
    sep="\t",
    nrows=5,
    encoding="utf-8"
)

print("Source 1:")
display(source1_sample)

print("Source 2:")
display(source2_sample)

print("Source 3:")
display(source3_sample)

Source 1:


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


Source 2:


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


Source 3:


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India


In [17]:
for name, df in {
    "Source 1": source1_sample,
    "Source 2": source2_sample,
    "Source 3": source3_sample
}.items():

    print(f"\n===== {name} =====")
    print("Columns:", df.columns.tolist())
    print("\nData types:")
    print(df.dtypes)
    print("\nMissing values:")
    print(df.isna().sum())
    print("\nDuplicate entity IDs:", df["entity_id"].duplicated().sum())


===== Source 1 =====
Columns: ['entity_id', 'business_name', 'business_address', 'country']

Data types:
entity_id           str
business_name       str
business_address    str
country             str
dtype: object

Missing values:
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

Duplicate entity IDs: 0

===== Source 2 =====
Columns: ['entity_id', 'business_name', 'business_address', 'country']

Data types:
entity_id           str
business_name       str
business_address    str
country             str
dtype: object

Missing values:
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

Duplicate entity IDs: 0

===== Source 3 =====
Columns: ['entity_id', 'business_name', 'business_address', 'country']

Data types:
entity_id           str
business_name       str
business_address    str
country             str
dtype: object

Missing values:
entity_id           0
business_name       0
busin

In [18]:
# Memory-safe profile of the full Source 1 dataset

source1_path = TRAIN_DIR / "train_source1.tsv"

total_rows = 0
missing_counts = pd.Series(dtype="int64")
entity_ids = set()
duplicate_ids = 0

for chunk in pd.read_csv(
    source1_path,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    total_rows += len(chunk)

    # Count missing values
    if missing_counts.empty:
        missing_counts = chunk.isna().sum()
    else:
        missing_counts = missing_counts.add(
            chunk.isna().sum(),
            fill_value=0
        )

    # Check duplicate entity IDs across the entire file
    for entity_id in chunk["entity_id"]:
        if entity_id in entity_ids:
            duplicate_ids += 1
        else:
            entity_ids.add(entity_id)

print("===== Source 1 Full Dataset =====")
print("Total rows:", total_rows)
print("\nMissing values:")
print(missing_counts.astype(int))
print("\nUnique entity IDs:", len(entity_ids))
print("Duplicate entity IDs:", duplicate_ids)

===== Source 1 Full Dataset =====
Total rows: 2206821

Missing values:
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

Unique entity IDs: 2206821
Duplicate entity IDs: 0


In [19]:
# Memory-safe profile of the full Source 2 dataset

source2_path = TRAIN_DIR / "train_source2.tsv"

total_rows = 0
missing_counts = pd.Series(dtype="int64")
entity_ids = set()
duplicate_ids = 0

for chunk in pd.read_csv(
    source2_path,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    total_rows += len(chunk)

    if missing_counts.empty:
        missing_counts = chunk.isna().sum()
    else:
        missing_counts = missing_counts.add(
            chunk.isna().sum(),
            fill_value=0
        )

    for entity_id in chunk["entity_id"]:
        if entity_id in entity_ids:
            duplicate_ids += 1
        else:
            entity_ids.add(entity_id)

print("===== Source 2 Full Dataset =====")
print("Total rows:", total_rows)
print("\nMissing values:")
print(missing_counts.astype(int))
print("\nUnique entity IDs:", len(entity_ids))
print("Duplicate entity IDs:", duplicate_ids)

===== Source 2 Full Dataset =====
Total rows: 5034616

Missing values:
entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

Unique entity IDs: 5034616
Duplicate entity IDs: 0


In [20]:
# Memory-safe profile of the full Source 3 dataset

source3_path = TRAIN_DIR / "train_source3.tsv"

total_rows = 0
missing_counts = pd.Series(dtype="int64")
entity_ids = set()
duplicate_ids = 0

for chunk in pd.read_csv(
    source3_path,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    total_rows += len(chunk)

    if missing_counts.empty:
        missing_counts = chunk.isna().sum()
    else:
        missing_counts = missing_counts.add(
            chunk.isna().sum(),
            fill_value=0
        )

    for entity_id in chunk["entity_id"]:
        if entity_id in entity_ids:
            duplicate_ids += 1
        else:
            entity_ids.add(entity_id)

print("===== Source 3 Full Dataset =====")
print("Total rows:", total_rows)
print("\nMissing values:")
print(missing_counts.astype(int))
print("\nUnique entity IDs:", len(entity_ids))
print("Duplicate entity IDs:", duplicate_ids)

===== Source 3 Full Dataset =====
Total rows: 5285603

Missing values:
entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64

Unique entity IDs: 5285603
Duplicate entity IDs: 0


In [21]:
# Inspect records with missing business names

print("===== Source 2: Missing business names =====")

for chunk in pd.read_csv(
    source2_path,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    missing = chunk[chunk["business_name"].isna()]
    if not missing.empty:
        display(missing)


print("===== Source 3: Missing business names =====")

for chunk in pd.read_csv(
    source3_path,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    missing = chunk[chunk["business_name"].isna()]
    if not missing.empty:
        display(missing)

===== Source 2: Missing business names =====


,entity_id,business_name,business_address,country
1068091,S2-982925237,NaN,"11 OX YOKE CIRCLE, EAST HAMPTON, CT",US


,entity_id,business_name,business_address,country
4065413,S2-599473918,NaN,"BROOKHAVEN, 7 LILAC LANE, NY",US


===== Source 3: Missing business names =====


,entity_id,business_name,business_address,country
160286,S3-391601813,NaN,"209 3rd Street, KY, Hardinsburg",US


...

,entity_id,business_name,business_address,country
3112931,S3-749220699,NaN,"7517 Oleander Avenue, Chicago, IL",US
3155293,S3-263823274,NaN,"519 Odeneal Street, Dallas, TX",US


,entity_id,business_name,business_address,country
3349157,S3-640593334,NaN,"261/12 F/F Amritpuri B Garhi Jharia Maria, Sou...",India


,entity_id,business_name,business_address,country
3697228,S3-207014979,NaN,"C-2 Mansarovar Riico Industrial Area, Jaipur, ...",India


,entity_id,business_name,business_address,country
4638561,S3-382185873,NaN,"Kh No. 138, 1St Floor, Vill: Nawada, Vipin Gar...",India


,entity_id,business_name,business_address,country
4939588,S3-761073618,NaN,"H.No. 25/B East Anandpuri, Patna, Bihar",India
4973878,S3-617170111,NaN,"120 Shore Drive, Jarvisburg, North Carolina",US


In [22]:
# Inspect records with missing business addresses

print("===== Source 2: Missing business addresses (first 20) =====")

count = 0

for chunk in pd.read_csv(
    source2_path,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    missing = chunk[chunk["business_address"].isna()]

    if not missing.empty:
        display(missing.head(20))
        count += len(missing)

        if count >= 20:
            break


print("===== Source 3: Missing business addresses (first 20) =====")

count = 0

for chunk in pd.read_csv(
    source3_path,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    missing = chunk[chunk["business_address"].isna()]

    if not missing.empty:
        display(missing.head(20))
        count += len(missing)

        if count >= 20:
            break

===== Source 2: Missing business addresses (first 20) =====


,entity_id,business_name,business_address,country
53,S2-187379771,"Guerra And Krueger Table, LLC",NaN,US
69,S2-751730544,L E Pierce Cal [PC],NaN,US
93,S2-162849450,Preferred Bay Montage Inc,NaN,US
195,S2-616836723,DELONG STRATEGIC READY,NaN,US
236,S2-39617527,Desert Vanguard ([Maintenance]),NaN,US
346,S2-582984116,Wildlife Sóciety,NaN,US
352,S2-762023797,Orellana Kensington,NaN,US
376,S2-682660453,rathi holdings,NaN,India
390,S2-752363850,Berry Saul,NaN,US
424,S2-14102665,"Thrasher, Wanner 5tenson LLC Center",NaN,US


===== Source 3: Missing business addresses (first 20) =====


,entity_id,business_name,business_address,country
1,S3-859268022,International South Consultants Private Ltd,NaN,India
52,S3-92001570,Davis Acquisition,NaN,US
105,S3-299576808,"Morales, Godina & Pánkow Associates LLC",NaN,US
110,S3-924073964,The Nancy Nancy Bunten Energy,NaN,US
125,S3-588120445,Anand Agro Private Enterprises,NaN,India
189,S3-32634960,Shyam Foundation Private Limited,NaN,India
190,S3-509442398,New Delhi Techn0logies Private Limited,NaN,India
231,S3-987949855,Southern பவர்,NaN,India
251,S3-301461097,Al Macrktig Pvt Ltd,NaN,India
302,S3-180062010,Beus Inc. Center,NaN,US


In [23]:
# Check duplicate business names in Source 2 and Source 3
# We only inspect the first few duplicated names.

print("===== Source 2: Duplicate business names =====")

duplicate_names_found = 0

for chunk in pd.read_csv(
    source2_path,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    duplicates = chunk[
        chunk["business_name"].notna()
        & chunk["business_name"].duplicated(keep=False)
    ]

    if not duplicates.empty:
        display(
            duplicates.sort_values("business_name").head(20)
        )
        duplicate_names_found += len(duplicates)

        if duplicate_names_found >= 20:
            break


print("\n===== Source 3: Duplicate business names =====")

duplicate_names_found = 0

for chunk in pd.read_csv(
    source3_path,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    duplicates = chunk[
        chunk["business_name"].notna()
        & chunk["business_name"].duplicated(keep=False)
    ]

    if not duplicates.empty:
        display(
            duplicates.sort_values("business_name").head(20)
        )
        duplicate_names_found += len(duplicates)

        if duplicate_names_found >= 20:
            break

===== Source 2: Duplicate business names =====


,entity_id,business_name,business_address,country
69245,S2-575194660,#newlife,"#36 MONTGOMERY ST, GOSHEN, NY",US
33337,S2-362368907,#newlife,"24 SPRINGVALE ROAD, NULL, CORTLANDT, NY",US
9696,S2-235786647,#newlife,"1848B NORTH BLVD, FAIRBORN, OH",US
84768,S2-459327705,24HR Homecare,"032543 BEYMER RD, EUGENE, OR",US
59618,S2-312985723,24HR Homecare,"GREECE, NY, 157 EVERWILD LN",US
65763,S2-862575276,24HR Locksmith,"ORCHARD ST, MUNDELEIN, IL",US
27126,S2-417579530,24HR Locksmith,"17820 STARS LN, MD, HAGERSTOWN",US
1498,S2-725343258,3-Corners,"3626 SEVERN RD, CLEVELAND HEIGHTS, OH",US
69216,S2-531731149,3-Corners,"3570 LOVELACEVILLE RD, PADUCAH, KY",US
8197,S2-264371358,6point.com,"10520 IH 10, BAYTOWN, TX",US



===== Source 3: Duplicate business names =====


,entity_id,business_name,business_address,country
69608,S3-475101048,#pediatricdental,"Iowa, 314 Johnson Srteet, Alta",US
18433,S3-172347033,#pediatricdental,"929 7th Avenue, City Of Wausau, Wisconsin",US
80943,S3-597738560,#physicaltherapy,"8424 Pettit Rd, Baker, Louisiana",US
85189,S3-838027852,#physicaltherapy,"12438 Lebanon Road, Mt Juliet, Tennessee",US
19143,S3-742705431,1 800,"6001 S St, Sacramento, California",US
49263,S3-752409718,1 800,"630 Chip Ct, Gurnee, Illinois",US
63528,S3-310858994,2-Brothers LLC,"353 Stmwer Ln, Unit 1E, Norawlk CDP, Ohio",US
68583,S3-201789260,2-Brothers LLC,"New York, 11215 Genesee St, Fl Floor 1, Alden",US
11724,S3-733756281,@pediatricdental,"North Carolina, Charlotte, 8004 Red Water Rd",US
79407,S3-219297475,@pediatricdental,"La Flolette, 113 Court Ln, Tennessee",US


In [24]:
# Check duplicate business addresses in Source 2 and Source 3

print("===== Source 2: Duplicate business addresses =====")

duplicate_addresses_found = 0

for chunk in pd.read_csv(
    source2_path,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    duplicates = chunk[
        chunk["business_address"].notna()
        & chunk["business_address"].duplicated(keep=False)
    ]

    if not duplicates.empty:
        display(
            duplicates.sort_values("business_address").head(20)
        )
        duplicate_addresses_found += len(duplicates)

        if duplicate_addresses_found >= 20:
            break


print("\n===== Source 3: Duplicate business addresses =====")

duplicate_addresses_found = 0

for chunk in pd.read_csv(
    source3_path,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    duplicates = chunk[
        chunk["business_address"].notna()
        & chunk["business_address"].duplicated(keep=False)
    ]

    if not duplicates.empty:
        display(
            duplicates.sort_values("business_address").head(20)
        )
        duplicate_addresses_found += len(duplicates)

        if duplicate_addresses_found >= 20:
            break

===== Source 2: Duplicate business addresses =====


,entity_id,business_name,business_address,country
35249,S2-863086267,Vidhi Commotrade Private,"# B2. 291, ST JOHNS ROAD, 9TH FLOOR SOBHA IVOR...",India
52787,S2-477125949,Vidhi Commotrade Private Ltd,"# B2. 291, ST JOHNS ROAD, 9TH FLOOR SOBHA IVOR...",India
59183,S2-835971466,Continental Precision Invesco,"##1202 OHIO, LA PORTE, IN",US
16484,S2-831707743,Continental (Precision),"##1202 OHIO, LA PORTE, IN",US
25514,S2-388887651,BLUE CLEANING SERVICE,"##401 97ND WAY, MESA FOUR PEAKS, AZ",US
44696,S2-478396541,Blue Cleaning Srbrvite,"##401 97ND WAY, MESA FOUR PEAKS, AZ",US
21927,S2-633542186,Center Gibbins Grand LLC,"##4200 CREIGHTON RD, null, HENRICO COUNTY, VA",US
30683,S2-258903708,Gibbins Grand Horse,"##4200 CREIGHTON RD, null, HENRICO COUNTY, VA",US
51951,S2-56228460,M/s INTERNET VISION PRIVATE LIMITED,"##8-A, NATIONAL HIGHWAY, OPP. SARVODAY HOTEL D...",India
81659,S2-371020357,M/s Internet Vision Limited Services,"##8-A, NATIONAL HIGHWAY, OPP. SARVODAY HOTEL D...",India



===== Source 3: Duplicate business addresses =====


,entity_id,business_name,business_address,country
45981,S3-47549836,B+ Atlanticus,"##10900 121, Frisco, Texas",US
56318,S3-412601735,B+ Atlanticus,"##10900 121, Frisco, Texas",US
53105,S3-882598109,Highland Productions Golden LLC,"##16 Foster Street, Danburytownship, Connecticut",US
82744,S3-358725216,Highland Golden Prodtucins LLC,"##16 Foster Street, Danburytownship, Connecticut",US
24547,S3-550490677,"Nguyen, Brixey & Australia Ashton LLC","##2017 Wheeler Ln, Heber Ity, Utah",US
6296,S3-609260854,Orbis Wheeler,"##2017 Wheeler Ln, Heber Ity, Utah",US
98171,S3-53348611,prime textile worldwide llc,"##3116 Winesap Road, Kingsport, Tennessee",US
13224,S3-686220330,Prime Textile Worldwide,"##3116 Winesap Road, Kingsport, Tennessee",US
39515,S3-460576539,Internal Medicine Clinic,"##35945 Dayspring St, Soldotna, Alaska",US
18421,S3-481339929,Internal Medicine Clinic,"##35945 Dayspring St, Soldotna, Alaska",US


In [25]:
# Inspect unusual characters and formatting in business names

print("===== Source 2: Sample business names =====")

source2_names = []

for chunk in pd.read_csv(
    source2_path,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    names = chunk["business_name"].dropna().astype(str)

    source2_names.extend(names.head(10).tolist())

    if len(source2_names) >= 50:
        break

for name in source2_names[:50]:
    print(repr(name))


print("\n===== Source 3: Sample business names =====")

source3_names = []

for chunk in pd.read_csv(
    source3_path,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    names = chunk["business_name"].dropna().astype(str)

    source3_names.extend(names.head(10).tolist())

    if len(source3_names) >= 50:
        break

for name in source3_names[:50]:
    print(repr(name))

===== Source 2: Sample business names =====
'राम मार्केटिंग प्राइवेट लिमिटेड'
'-- Holloway Peak Inc Seafood'
'आदित्य प्रॉपर्टीज एलएलपी'
'Summit Inc'
'Delta Tetlecommunication Inc'
'Lee and Lawson'
'SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPORATION | www.shivshakti.com'
'Shree Infracon Private Ltd'
'Chavira Platinum Chimera LLC'
'FOUNDATION EXCEL AGENCY PRIVATE  LIMITED'
'Rapid Institute of Teclnholeoeg'
'Southern Logistics'
'AFP (Defi)'
'kv patriot series llc'
'Lee & Woodley Center'
'Raleigh Mnemordal Fund LP'
'Downtown-Pub!'
'Sayers  Green Labs North'
'ONCOLOGY MEDICINE MEDICINE ÍNC'
'VITHALDAS DEGREE (INDIA) GROUP PRIVATE LIMITED'
'PW Castings Private'
'YR QUALITY ONCOLOGY LLC'
'Raimondi Bgrse L.L.C.'
'Solutions At Travels Privioate Limited'
'Glyphent Brewing'
'FIRST MATERIALS LABS-INC'
'W A Gbmay Biologics Corp'
'Advaita (india) Infratech Pvt Ltd'
'Hicks and Ray Construction'
'Beth Fellowship (Co)'
'Bright Vidyalaya Enterprises Private'
'Bryant Agriculture Girard &'
'Shri Internat

In [34]:
# Basic text normalization for entity resolution

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Replace punctuation/symbols while preserving Unicode letters
    text = "".join(
        char if (
            unicodedata.category(char).startswith("L")
            or unicodedata.category(char).startswith("M")
            or unicodedata.category(char).startswith("N")
            or char.isspace()
        ) else " "
        for char in text
    )

    # Normalize multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [35]:
test_names = [
    "A CURE",
    "A Cure",
    "Renfro's Yoga, Ínc",
    "Renfro's Yoga, Inc",
    "FOUNDATION EXCEL AGENCY PRIVATE  LIMITED",
    "राम मार्केटिंग प्राइवेट लिमिटेड",
    "Downtown-Pub!",
    "SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPORATION | https://www.shivshakti.com"
]

for name in test_names:
    print("Original :", name)
    print("Normalized:", normalize_text(name))
    print()

Original : A CURE
Normalized: a cure

Original : A Cure
Normalized: a cure

Original : Renfro's Yoga, Ínc
Normalized: renfro s yoga ínc

Original : Renfro's Yoga, Inc
Normalized: renfro s yoga inc

Original : FOUNDATION EXCEL AGENCY PRIVATE  LIMITED
Normalized: foundation excel agency private limited

Original : राम मार्केटिंग प्राइवेट लिमिटेड
Normalized: राम मार्केटिंग प्राइवेट लिमिटेड

Original : Downtown-Pub!
Normalized: downtown pub

Original : SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPORATION | https://www.shivshakti.com
Normalized: shivshakti vidyalaya vidyalaya overseas corporation



In [43]:
sample_addresses = source2_sample.copy()

sample_addresses["business_address_normalized"] = (
    sample_addresses["business_address"]
    .apply(normalize_text)
)

display(
    sample_addresses[
        ["business_address", "business_address_normalized"]
    ]
)

,business_address,business_address_normalized
0,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",kh no 570 13 new delhi west delhi delhi
1,"105 ELM ST, MORGANTON, NC",105 elm st morganton nc
2,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",g 3 571 gulmohar colony bhopal madhya pradesh
3,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",greensboro nc 19 1 2 stardust trail
4,"914 PIERPONT AVE, CLEVELAND, OH",914 pierpont ave cleveland oh


In [47]:
# Inspect country values

print("Source 2 countries:")
print(source2_sample["country"].tolist())

print("\nSource 3 countries:")
print(source3_sample["country"].tolist())

Source 2 countries:
['India', 'US', 'India', 'US', 'US']

Source 3 countries:
['US', 'India', 'US', 'US', 'India']


In [48]:
print("Source 3 countries:")
print(source3_sample["country"].tolist())

Source 3 countries:
['US', 'India', 'US', 'US', 'India']


In [51]:
# Create processed data folders

PROCESSED_TRAIN_DIR = PROJECT_ROOT / "data" / "processed" / "train"
PROCESSED_TEST_DIR = PROJECT_ROOT / "data" / "processed" / "test"

PROCESSED_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_TEST_DIR.mkdir(parents=True, exist_ok=True)

print("Processed train folder:", PROCESSED_TRAIN_DIR)
print("Processed test folder:", PROCESSED_TEST_DIR)

Processed train folder: c:\Users\nihar\OneDrive\Attachments\AmazonML-26\data\processed\train
Processed test folder: c:\Users\nihar\OneDrive\Attachments\AmazonML-26\data\processed\test


In [53]:
# Process Source 1 in chunks

source1_input = TRAIN_DIR / "train_source1.tsv"
source1_output = PROCESSED_TRAIN_DIR / "train_source1_cleaned.tsv"

# Remove old output if it exists
if source1_output.exists():
    source1_output.unlink()

first_chunk = True

for chunk in pd.read_csv(
    source1_input,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    # Normalize business name
    chunk["business_name_normalized"] = (
        chunk["business_name"].apply(normalize_text)
    )

    # Normalize business address
    chunk["business_address_normalized"] = (
        chunk["business_address"].apply(normalize_text)
    )

    # Keep original columns + normalized columns
    chunk.to_csv(
        source1_output,
        sep="\t",
        index=False,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        encoding="utf-8"
    )

    first_chunk = False

print("Source 1 cleaning completed.")
print("Saved to:", source1_output)

Source 1 cleaning completed.
Saved to: c:\Users\nihar\OneDrive\Attachments\AmazonML-26\data\processed\train\train_source1_cleaned.tsv


In [55]:
# Process Source 2 in chunks

source2_input = TRAIN_DIR / "train_source2.tsv"
source2_output = PROCESSED_TRAIN_DIR / "train_source2_cleaned.tsv"

if source2_output.exists():
    source2_output.unlink()

first_chunk = True

for chunk in pd.read_csv(
    source2_input,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    chunk["business_name_normalized"] = (
        chunk["business_name"].apply(normalize_text)
    )

    chunk["business_address_normalized"] = (
        chunk["business_address"].apply(normalize_text)
    )

    chunk.to_csv(
        source2_output,
        sep="\t",
        index=False,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        encoding="utf-8"
    )

    first_chunk = False

print("Source 2 cleaning completed.")
print("Saved to:", source2_output)

Source 2 cleaning completed.
Saved to: c:\Users\nihar\OneDrive\Attachments\AmazonML-26\data\processed\train\train_source2_cleaned.tsv


In [56]:
# Process Source 3 in chunks

source3_input = TRAIN_DIR / "train_source3.tsv"
source3_output = PROCESSED_TRAIN_DIR / "train_source3_cleaned.tsv"

if source3_output.exists():
    source3_output.unlink()

first_chunk = True

for chunk in pd.read_csv(
    source3_input,
    sep="\t",
    encoding="utf-8",
    chunksize=100_000
):
    chunk["business_name_normalized"] = (
        chunk["business_name"].apply(normalize_text)
    )

    chunk["business_address_normalized"] = (
        chunk["business_address"].apply(normalize_text)
    )

    chunk.to_csv(
        source3_output,
        sep="\t",
        index=False,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        encoding="utf-8"
    )

    first_chunk = False

print("Source 3 cleaning completed.")
print("Saved to:", source3_output)

Source 3 cleaning completed.
Saved to: c:\Users\nihar\OneDrive\Attachments\AmazonML-26\data\processed\train\train_source3_cleaned.tsv


In [57]:
# Verify cleaned training datasets

cleaned_files = [
    PROCESSED_TRAIN_DIR / "train_source1_cleaned.tsv",
    PROCESSED_TRAIN_DIR / "train_source2_cleaned.tsv",
    PROCESSED_TRAIN_DIR / "train_source3_cleaned.tsv"
]

for file_path in cleaned_files:
    print("\n==============================")
    print("File:", file_path.name)
    print("Exists:", file_path.exists())

    if file_path.exists():
        sample = pd.read_csv(
            file_path,
            sep="\t",
            encoding="utf-8",
            nrows=5
        )

        print("Columns:", sample.columns.tolist())
        print("Sample rows:", len(sample))
        display(sample)


File: train_source1_cleaned.tsv
Exists: True
Columns: ['entity_id', 'business_name', 'business_address', 'country', 'business_name_normalized', 'business_address_normalized']
Sample rows: 5


,entity_id,business_name,business_address,country,business_name_normalized,business_address_normalized
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop,1795 westchester drive high point nc
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money,17560 ellis road tahlequah ok
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,b retail inc,1712 montebello avenue phoenix az
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,christ chapel,2100 cameron drive unit apartment g dundalk md
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,prabhav business center,797 lake town block a kolkata howrah west bengal



File: train_source2_cleaned.tsv
Exists: True
Columns: ['entity_id', 'business_name', 'business_address', 'country', 'business_name_normalized', 'business_address_normalized']
Sample rows: 5


,entity_id,business_name,business_address,country,business_name_normalized,business_address_normalized
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,राम मार्केटिंग प्राइवेट लिमिटेड,kh no 570 13 new delhi west delhi delhi
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,holloway peak inc seafood,105 elm st morganton nc
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,आदित्य प्रॉपर्टीज एलएलपी,g 3 571 gulmohar colony bhopal madhya pradesh
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,summit inc,greensboro nc 19 1 2 stardust trail
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,delta tetlecommunication inc,914 pierpont ave cleveland oh



File: train_source3_cleaned.tsv
Exists: True
Columns: ['entity_id', 'business_name', 'business_address', 'country', 'business_name_normalized', 'business_address_normalized']
Sample rows: 5


,entity_id,business_name,business_address,country,business_name_normalized,business_address_normalized
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US,wilfordhancock com,mack rd haltom city texas
1,S3-859268022,International South Consultants Private Ltd,NaN,India,international south consultants private ltd,NaN
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US,llc moncada léarning center,5780 fawn ct fort worth texas
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US,moyna s coffee,1 ivanhoe ave po box 6009 cincinnati ohio
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India,pvt efs print ventures ltd,door no 183 41st cross 22nd main 9th block jay...


In [58]:
# Verify row counts of cleaned files

for file_path in cleaned_files:
    row_count = 0

    for chunk in pd.read_csv(
        file_path,
        sep="\t",
        encoding="utf-8",
        chunksize=100_000
    ):
        row_count += len(chunk)

    print(file_path.name, "→", row_count, "rows")

train_source1_cleaned.tsv → 2206821 rows
train_source2_cleaned.tsv → 5034616 rows
train_source3_cleaned.tsv → 5285603 rows


In [59]:
print(file_path.name, "→", row_count, "rows")

train_source3_cleaned.tsv → 5285603 rows
